# Ground Condition Predictor — **ground-up v11** (grass / loose / paved)

Features engineered **from the raw signals**, not inherited from the accumulated pipeline.
Design corrected after diagnosis: accelerometer features (jerk on accel-Y, spectral on |accel|)
ported faithfully, step-to-step foot-tilt variability kept, and **per-sensor pressure ratios
dropped** (they act as a person fingerprint and hurt cross-user generalisation).

| Raw label | Class |
|---|---|
| grass | `grass` |
| dirt, compact ground | `loose` |
| concrete, asphalt, exposed aggregate, brick | `paved` |
| transition, wait | *excluded* |

30 features, no per-sensor ratios. Same section layout as the accumulated pipeline, minus the
prediction cells. Flip `CV_GROUP` between `source_recording` (new session) and `source_user`
(new person) for the two honest evaluations.

In [5]:
import os, glob, numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
try:
    from xgboost import XGBClassifier; HAS_XGB = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier; HAS_XGB = False

# ---- paths ----
RAW_ROOT   = r'C:\Users\Sam\WalkSensePlace\Individual Users'          # <-- SET THIS to your recordings folder (absolute path ok; searched recursively)
CHARTS_DIR = 'Charts_Graphs'; os.makedirs(CHARTS_DIR, exist_ok=True)

# ---- signal / layout ----
SAMPLING_HZ  = 62.5
ALL_SENSORS  = [f'pressure_{i:02d}' for i in range(1, 13)]
HEEL_SENSORS = ['pressure_08', 'pressure_11']
FORE_SENSORS = ['pressure_02', 'pressure_04', 'pressure_05', 'pressure_06', 'pressure_09']
BASE_COLS    = set(['sole_id','timestamp','accel_x','accel_y','accel_z','gyro_x','gyro_y','gyro_z',
                    'magn_x','magn_y','magn_z','corrupt'] + ALL_SENSORS)

# ---- labelling ----
SURFACE_MAP = {'grass':'grass','dirt':'loose','compact ground':'loose','compacted ground':'loose',
               'concrete':'paved','asphalt':'paved','exposed aggregate':'paved','brick':'paved'}
EXCLUDE_TAGS   = {'transition','wait'}     # carve out, never absorb into a neighbour
SURFACE_COLORS = {'grass':'#4CAF50','loose':'#8D6E63','paved':'#607D8B','unknown':'#9E9E9E'}
FOCUS_CLASS    = 'loose'

# ---- evaluation ----
CV_GROUP = 'source_recording'    # 'source_recording' (new session) | 'source_user' (new person)
STEP_MIN, STEP_MAX = 15, 120     # samples per step (~0.24-1.9 s at 62.5 Hz)

def make_model():
    if HAS_XGB:
        return XGBClassifier(n_estimators=350, learning_rate=0.05, max_depth=5, subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=3, eval_metric='mlogloss',
                             random_state=42, n_jobs=-1, verbosity=0)
    return HistGradientBoostingClassifier(max_depth=5, learning_rate=0.05, max_iter=350, random_state=42)
print('XGBoost available:', HAS_XGB)

XGBoost available: True


## 2. Build features from the raw recordings

Parses surface markers per row (each marker runs until the next; `transition`/`wait` carve out an
excluded region), splits by sole, detects steps as peaks of total pressure, and extracts the 30
features per single-surface step. No pre-compiled per-surface CSVs — everything is built here.

In [6]:
def label_rows(df):
    """Per-row surface from 'x' markers. Each marker runs to the next; EXCLUDE_TAGS -> None
    (carved out, not absorbed)."""
    df = df.sort_values('timestamp').reset_index(drop=True); df['surf'] = None; ev = []
    for c in df.columns:
        if c in BASE_COLS: continue
        for idx in df.index[df[c].astype(str).str.strip() == 'x']:
            ev.append((int(idx), None if c in EXCLUDE_TAGS else SURFACE_MAP.get(c)))
    ev = [e for e in ev if (e[1] is not None) or True]  # keep excludes as boundaries
    # build events incl excludes (as sentinel) so a marker still bounds the previous surface
    ev2 = []
    for c in df.columns:
        if c in BASE_COLS: continue
        tag = '__EXC__' if c in EXCLUDE_TAGS else SURFACE_MAP.get(c)
        if tag is None: continue
        for idx in df.index[df[c].astype(str).str.strip() == 'x']:
            ev2.append((int(idx), tag))
    ev2.sort()
    for k,(i0,lab) in enumerate(ev2):
        i1 = ev2[k+1][0] if k+1 < len(ev2) else len(df)
        df.loc[i0:i1-1,'surf'] = (None if lab == '__EXC__' else lab)
    return df

def _spec_features(sig, fs=SAMPLING_HZ, hf_cut=5.0):
    """Spectral centroid, high-freq power fraction, normalised spectral entropy (amplitude-invariant)."""
    x = np.asarray(sig, float); n = len(x)
    if n < 8 or np.allclose(x, x[0]): return 0.0, 0.0, 0.0
    t = np.arange(n); x = x - np.polyval(np.polyfit(t, x, 1), t)
    P = np.abs(np.fft.rfft(x))**2; f = np.fft.rfftfreq(n, 1.0/fs); tot = P.sum()
    if tot <= 0: return 0.0, 0.0, 0.0
    p = P/tot; p = p[p > 0]
    return (float((f*P).sum()/tot), float(P[f >= hf_cut].sum()/tot),
            float(-(p*np.log(p)).sum()/np.log(len(P))) if len(P) > 1 else 0.0)

def _roughness(sig):
    """Zero-crossing rate + jerk-to-signal RMS ratio (amplitude-invariant)."""
    x = np.asarray(sig, float); n = len(x)
    if n < 4: return 0.0, 0.0
    xd = x - x.mean(); zcr = np.sum(np.diff(np.sign(xd)) != 0)/max(n-1, 1)
    return float(zcr), float(np.sqrt(np.mean(np.diff(x)**2))/(np.sqrt(np.mean(xd**2))+1e-9))

def _pent(v):
    """Normalised Shannon entropy of a 12-sensor pressure vector (0=one region, 1=even)."""
    v = np.clip(v, 0, None); t = v.sum()
    if t <= 0: return 0.0
    q = v/t; q = q[q > 0]; return float(-(q*np.log(q)).sum()/np.log(12))

def _rs(sig, L=50):
    xs = np.linspace(0, 1, len(sig)); return np.interp(np.linspace(0, 1, L), xs, sig)

def detect_steps(tot):
    mx = np.nanmax(tot) if np.nanmax(tot) > 0 else 1.0
    pk, _ = find_peaks(tot, distance=25, prominence=mx*0.15); return pk
print('helpers defined')

helpers defined


In [7]:
def extract_sole_steps(s):
    """Feature rows for one sole's dataframe (must have 'surf'). Skip-safe; single-surface steps only."""
    s = s.reset_index(drop=True)
    if not len(s): return []
    P    = s[ALL_SENSORS].apply(pd.to_numeric, errors='coerce').fillna(0).values
    tot  = P.sum(1)
    heel = s[HEEL_SENSORS].apply(pd.to_numeric, errors='coerce').sum(1).values
    fore = s[FORE_SENSORS].apply(pd.to_numeric, errors='coerce').sum(1).values
    ax = pd.to_numeric(s['accel_x'],errors='coerce').fillna(0).values
    ay = pd.to_numeric(s['accel_y'],errors='coerce').fillna(0).values
    az = pd.to_numeric(s['accel_z'],errors='coerce').fillna(0).values
    G  = s[['gyro_x','gyro_y','gyro_z']].apply(pd.to_numeric,errors='coerce').fillna(0).values
    Gc = G - np.median(G, 0)
    amag = np.sqrt(ax**2+ay**2+az**2); gmag = np.sqrt((Gc**2).sum(1)); surf = s['surf'].values
    tA = np.degrees(np.arctan2(ax, np.hypot(ay, az))); tB = np.degrees(np.arctan2(az, np.hypot(ax, ay)))
    pk = detect_steps(tot); out = []; order = 0
    for i in range(len(pk)-1):
        a, b = pk[i], pk[i+1]; n = b-a
        if n < STEP_MIN or n > STEP_MAX: continue
        u = pd.unique(surf[a:b][pd.notna(surf[a:b])])
        if len(u) != 1 or u[0] not in SURFACE_MAP.values(): continue
        Pw = P[a:b]; totw = tot[a:b]; sc = totw.mean()+1e-9; wam = amag[a:b]; wgm = gmag[a:b]; way = ay[a:b]
        totn = _rs(totw)/sc; foren = _rs(fore[a:b])/sc; heeln = _rs(heel[a:b])/sc
        m = int(np.argmin(wgm)); lo, hi = max(0,m-3), min(n,m+4)
        ff = Pw[lo:hi]; ffm = ff.mean(0) if len(ff) else Pw.mean(0)
        Qn = Pw/(Pw.sum(1,keepdims=True)+1e-9)
        st = np.where(wgm <= np.percentile(wgm,25))[0]; st = st if len(st) >= 3 else np.array([int(np.argmin(wgm))])
        a_cent, a_hf, a_ent = _spec_features(wam)
        _, g_hf, _          = _spec_features(wgm)
        a_zcr, a_jerk       = _roughness(way)      # accel-Y axis (the discriminative one)
        out.append({
            # magnitude (carries grass; within-user loose slightly > paved)
            'mean_total':float(totw.mean()),'peak_total':float(totw.max()),
            'mean_fore':float(fore[a:b].mean()),'mean_heel':float(heel[a:b].mean()),
            'heel_fore_ratio':float(heel[a:b].mean()/(fore[a:b].mean()+1e-9)),
            # normalised step shape
            'dip_depth':float(1 - totn[20:34].min()/(totn.max()+1e-9)),
            'load_first_frac':float(totn[:25].sum()/(totn.sum()+1e-9)),
            'push_peak':float(totn[38:].max()),'load_slope':float(np.max(np.diff(totn[:25]))),
            'unload_slope':float(np.min(np.diff(totn[20:40]))),
            'fore_early_frac':float(foren[:20].sum()/(foren.sum()+1e-9)),
            'fore_heel_toff':float((np.argmax(foren)-np.argmax(heeln))/50),
            'stance_frac':float((totw > 0.5*sc).mean()),'step_dur':float(n/SAMPLING_HZ),
            # pressure-distribution entropy / spread (flat-foot + stance)
            'spatial_entropy_ff':_pent(ffm),'spatial_entropy_stance':_pent(Pw.mean(0)),
            'participation_ff':float((1/np.sum((ffm/(ffm.sum()+1e-9))**2))/12),
            'active_sensors':float((ffm > 0.1*ffm.max()).sum() if ffm.max() > 0 else 0),
            # spatial dynamics
            'cop_wander':float(np.mean(np.abs(np.diff(Qn,axis=0)).sum(1))) if n > 1 else 0.0,
            'press_jitter':float(np.mean(np.std(np.diff(Pw,n=2,axis=0),axis=0))/sc) if n > 2 else 0.0,
            # accelerometer (ported faithfully from the accumulated pipeline)
            'accel_spec_centroid':a_cent,'accel_hf':a_hf,'accel_spec_entropy':a_ent,
            'accel_zcr':a_zcr,'accel_jerk_ratio':a_jerk,'accel_mag_strike':float(np.max(wam[:max(1,n//3)])),
            # gyro
            'gyro_hf':g_hf,'gyro_peak_norm':float(wgm.max()/(np.median(wgm)+1e-9)),
            # for step-to-step tilt variability (finished after grouping)
            'zA':float(np.mean(tA[a:b][st])),'zB':float(np.mean(tB[a:b][st])),
            'order':order,'surface':u[0]})
        order += 1
    return out
print('feature extractor defined  (NO per-sensor pressure ratios)')

feature extractor defined  (NO per-sensor pressure ratios)


In [8]:
# robust recursive discovery of recording CSVs under RAW_ROOT
FILES = sorted(glob.glob(os.path.join(RAW_ROOT, '**', '*.csv'), recursive=True))
if not FILES:
    raise FileNotFoundError(
        f"No CSVs found under RAW_ROOT={RAW_ROOT!r} (cwd={os.getcwd()!r}).\n"
        f"Set RAW_ROOT (in the config cell) to the folder that holds your recordings; "
        f"it may be an absolute path and can be nested, e.g. r'C:\\Users\\Sam\\...\\Individual Users'.")
print(f'{len(FILES)} recording files found under {RAW_ROOT!r}')

rows = []
for f in FILES:
    user = os.path.basename(os.path.dirname(f)); rec = os.path.basename(f)
    try:
        df = label_rows(pd.read_csv(f, low_memory=False))
    except Exception as e:
        print(f'  [skip] {rec}: {e}'); continue
    for sole in sorted(pd.to_numeric(df.get('sole_id', pd.Series(dtype=float)), errors='coerce').dropna().unique()):
        for r in extract_sole_steps(df[df['sole_id'] == sole]):
            rows.append({**r, 'source_user':user, 'source_recording':rec, 'sole':int(sole)})

if not rows:
    raise ValueError(
        "Files were found but no steps were extracted. Check that: (a) 'sole_id' holds the foot ids, "
        "(b) surface-marker columns match SURFACE_MAP keys (e.g. grass/dirt/concrete/asphalt/brick), "
        "(c) the 12 pressure_XX columns are present. "
        f"First file columns: {list(pd.read_csv(FILES[0], nrows=1).columns)}")

D = pd.DataFrame(rows).sort_values(['source_recording','sole','order']).reset_index(drop=True)
# step-to-step foot-tilt variability (rolling 3-step std within each recording+sole), then drop absolutes
for col, nm in [('zA','tiltA_s2s'), ('zB','tiltB_s2s')]:
    D[nm] = D.groupby(['source_recording','sole'])[col].transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0))
D = D.drop(columns=['zA','zB','sole','order'])
print(f'{len(D)} steps  |  surfaces {D.surface.value_counts().to_dict()}')
print(f'users {sorted(D.source_user.unique())}  |  recordings {D.source_recording.nunique()}')
print('steps per user x surface:')
print(D.pivot_table(index='source_user', columns='surface', values='mean_total', aggfunc='count', fill_value=0).to_string())

35 recording files found under 'C:\\Users\\Sam\\WalkSensePlace\\Individual Users'
12110 steps  |  surfaces {'paved': 7858, 'grass': 2728, 'loose': 1524}
users ['Catherine', 'Kristian', 'Sam']  |  recordings 35
steps per user x surface:
surface      grass  loose  paved
source_user                     
Catherine      199     64    909
Kristian      1874    925   3578
Sam            655    535   3371


## 3. Feature matrix & cross-validation setup

`CV_GROUP` selects the honest test: `source_recording` (held-out session, known person) or `source_user` (held-out person). Folds auto-shrink to the rarest class's group count.

In [9]:
FEATURES = [c for c in D.columns if c not in ['surface','source_user','source_recording']]
X = D[FEATURES].fillna(0).values
le = LabelEncoder(); y = le.fit_transform(D['surface'].values); classes = le.classes_
groups = D[CV_GROUP].values
gpc = D.groupby('surface')[CV_GROUP].nunique()
N_SPLITS = int(max(2, min(5, gpc.min())))
print(f'Feature matrix: {X.shape[0]} steps x {X.shape[1]} features')
print(f'Classes: {list(classes)}   counts: {D.surface.value_counts().to_dict()}')
print(f'\nGrouped CV by "{CV_GROUP}" - distinct groups per class:'); print(gpc.to_string())
print(f'N_SPLITS = {N_SPLITS}')
FOCUS_IDX = int(np.where(classes == FOCUS_CLASS)[0][0]) if FOCUS_CLASS in classes else 0

Feature matrix: 12110 steps x 30 features
Classes: ['grass', 'loose', 'paved']   counts: {'paved': 7858, 'grass': 2728, 'loose': 1524}

Grouped CV by "source_recording" - distinct groups per class:
surface
grass    20
loose    13
paved    29
N_SPLITS = 5


## 4. Honest evaluation — leakage vs difficulty

Shuffled (leaky) vs grouped (honest) vs grouped+class-weight vs a two-stage grass-then-rest model. Read the focus-class and macro F1 across the rows.

In [10]:
def oof_predict(Xf, yf, grp, spl, weight=False):
    oof = np.empty(len(yf), int)
    for tr, te in spl.split(Xf, yf, grp):
        m = make_model()
        sw = compute_sample_weight('balanced', yf[tr]) if weight else None
        if HAS_XGB: m.fit(Xf[tr], yf[tr], sample_weight=sw)
        else:       m.fit(Xf[tr], yf[tr], sample_weight=sw)
        oof[te] = m.predict(Xf[te])
    return oof

def scores(yf, oof):
    f = f1_score(yf, oof, average=None) * 100
    return f[FOCUS_IDX], f1_score(yf, oof, average='macro')*100, accuracy_score(yf, oof)*100

sk  = StratifiedKFold(N_SPLITS, shuffle=True, random_state=42)
sgk = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=42)
rowsr = []
rowsr.append(('Shuffled (leaky)',      scores(y, oof_predict(X, y, groups, sk))))
rowsr.append(('Grouped (no weight)',   scores(y, oof_predict(X, y, groups, sgk))))
rowsr.append(('Grouped + focus weight',scores(y, oof_predict(X, y, groups, sgk, weight=True))))
# two-stage: grass-vs-rest, then loose-vs-paved
if set(['grass','loose','paved']) <= set(classes):
    oof2 = np.empty(len(y), int)
    for tr, te in sgk.split(X, y, groups):
        yg = (y != np.where(classes=='grass')[0][0]).astype(int)   # 1 = non-grass
        m1 = make_model(); m1.fit(X[tr], yg[tr], sample_weight=compute_sample_weight('balanced', yg[tr]))
        nz = yg[tr] == 1
        m2 = make_model(); m2.fit(X[tr][nz], y[tr][nz], sample_weight=compute_sample_weight('balanced', y[tr][nz]))
        p1 = m1.predict(X[te]); out = np.where(p1==0, np.where(classes=='grass')[0][0], m2.predict(X[te])); oof2[te]=out
    rowsr.append(('Two-stage grouped+weight', scores(y, oof2)))
print(f'{"Setup":26s}{FOCUS_CLASS+" F1":>10s}{"Macro F1":>10s}{"Accuracy":>10s}')
for nm,(ff,mf,ac) in rowsr:
    print(f'{nm:26s}{ff:9.1f}%{mf:9.1f}%{ac:9.1f}%')

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got [1 2]

## 5. Main model — classification report & confusion matrix

Grouped, class-weighted (the honest deployable setting under the current `CV_GROUP`).

In [ ]:
oof = oof_predict(X, y, groups, StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=42), weight=True)
print(classification_report(y, oof, target_names=classes, digits=3))
cm = confusion_matrix(y, oof)
fig, ax = plt.subplots(figsize=(5.2,4.4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes); ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
ax.set_xlabel('predicted'); ax.set_ylabel('actual'); ax.set_title(f'Confusion ({CV_GROUP}-grouped)', fontweight='bold')
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j,i,cm[i,j],ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.tight_layout(); plt.savefig(os.path.join(CHARTS_DIR,'groundup_confusion.png'),dpi=140,bbox_inches='tight'); plt.show()

## 6. 1D-CNN on raw step windows (optional)

Lets a convolutional net learn its own features from the raw multi-channel step windows, evaluated with the same grouping. Requires `torch`; if absent, the cell reports so and is skipped.

In [ ]:
try:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False
    print('torch not available - skipping CNN. (Run raw_window_1dcnn.py in a torch env.)')

if HAS_TORCH:
    L = 40
    def _rsw(sig): xs=np.linspace(0,1,len(sig)); return np.interp(np.linspace(0,1,L),xs,sig)
    Xc, yc, gc = [], [], []
    for f in sorted(glob.glob(os.path.join(RAW_ROOT,'**','*.csv'), recursive=True)):
        user=os.path.basename(os.path.dirname(f)); rec=os.path.basename(f); df=label_rows(pd.read_csv(f,low_memory=False))
        for sole in [1,2]:
            s=df[df['sole_id']==sole].reset_index(drop=True)
            if not len(s): continue
            P=s[ALL_SENSORS].apply(pd.to_numeric,errors='coerce').fillna(0).values; tot=P.sum(1)
            heel=s[HEEL_SENSORS].apply(pd.to_numeric,errors='coerce').sum(1).values; fore=s[FORE_SENSORS].apply(pd.to_numeric,errors='coerce').sum(1).values
            am=np.sqrt(s[['accel_x','accel_y','accel_z']].apply(pd.to_numeric,errors='coerce').fillna(0).pow(2).sum(1).values)
            gm=np.sqrt(s[['gyro_x','gyro_y','gyro_z']].apply(pd.to_numeric,errors='coerce').fillna(0).pow(2).sum(1).values)
            surf=s['surf'].values; pk=detect_steps(tot)
            for i in range(len(pk)-1):
                a,b=pk[i],pk[i+1]
                if b-a<STEP_MIN or b-a>STEP_MAX: continue
                u=pd.unique(surf[a:b][pd.notna(surf[a:b])])
                if len(u)!=1 or u[0] not in SURFACE_MAP.values(): continue
                sc=tot[a:b].mean()+1e-9
                Xc.append(np.stack([_rsw(tot[a:b])/sc,_rsw(heel[a:b])/sc,_rsw(fore[a:b])/sc,
                                    _rsw(am[a:b])/(np.median(am[a:b])+1e-9),_rsw(gm[a:b])/(np.median(gm[a:b])+1e-9)]))
                yc.append(u[0]); gc.append(rec if CV_GROUP=='source_recording' else user)
    Xc=np.array(Xc,np.float32); yc=np.array(yc); gc=np.array(gc)
    lec=LabelEncoder(); yci=lec.fit_transform(yc); ncl=len(lec.classes_)
    class Net(nn.Module):
        def __init__(s):
            super().__init__()
            s.c=nn.Sequential(nn.Conv1d(5,32,5,padding=2),nn.BatchNorm1d(32),nn.ReLU(),nn.MaxPool1d(2),
                              nn.Conv1d(32,64,5,padding=2),nn.BatchNorm1d(64),nn.ReLU(),nn.AdaptiveAvgPool1d(1))
            s.h=nn.Sequential(nn.Flatten(),nn.Dropout(0.3),nn.Linear(64,ncl))
        def forward(s,x): return s.h(s.c(x))
    oofc=np.empty(len(yci),int)
    for tr,te in StratifiedGroupKFold(N_SPLITS,shuffle=True,random_state=42).split(Xc,yci,gc):
        mu,sd=Xc[tr].mean((0,2),keepdims=True),Xc[tr].std((0,2),keepdims=True)+1e-6
        cw=torch.tensor([len(yci[tr])/(ncl*max((yci[tr]==k).sum(),1)) for k in range(ncl)],dtype=torch.float32)
        net=Net(); opt=torch.optim.Adam(net.parameters(),1e-3,weight_decay=1e-4); lf=nn.CrossEntropyLoss(weight=cw)
        dl=DataLoader(TensorDataset(torch.tensor((Xc[tr]-mu)/sd),torch.tensor(yci[tr])),batch_size=128,shuffle=True)
        net.train()
        for _ in range(40):
            for xb,yb in dl: opt.zero_grad(); lf(net(xb),yb).backward(); opt.step()
        net.eval()
        with torch.no_grad(): oofc[te]=net(torch.tensor((Xc[te]-mu)/sd)).argmax(1).numpy()
    fc=f1_score(yci,oofc,average=None)*100
    print(f'1D-CNN {CV_GROUP}-grouped: acc {accuracy_score(yci,oofc)*100:.0f}%  ' +
          '  '.join(f'{lec.classes_[k]} {fc[k]:.0f}%' for k in range(ncl)) + f'  macro {fc.mean():.0f}%')

## 7. Feature retention + Hardness / Evenness scales

RandomForest importance (which features to keep), a feature-by-surface heatmap, and two per-step scales — hardness (supervised P(firm)) and evenness (irregularity composite) — with the surface populations along them.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
full = D.copy()
# (A) importance / retention
rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(full[FEATURES].fillna(0), full['surface'])
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
KEEP = imp[imp > imp.max()*0.05].index.tolist()
print(f'Retain {len(KEEP)}/{len(FEATURES)} (>5% of max importance)'); print(imp.head(12).round(4).to_string())
fig,ax=plt.subplots(figsize=(8,6)); top=imp.head(18)[::-1]
ax.barh(range(len(top)),top.values,color='#3F51B5'); ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index,fontsize=9)
ax.set_title('Feature importance (RandomForest)',fontweight='bold'); plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR,'groundup_feature_importance.png'),dpi=140,bbox_inches='tight'); plt.show()
# (B) feature x surface heatmap
topf=imp.head(16).index.tolist(); Z=(full[topf]-full[topf].mean())/full[topf].std(); H=Z.groupby(full['surface']).mean().T
fig,ax=plt.subplots(figsize=(1.6*len(classes)+4,7)); im=ax.imshow(H.values,cmap='RdBu_r',vmin=-1.2,vmax=1.2,aspect='auto')
ax.set_xticks(range(H.shape[1])); ax.set_xticklabels(H.columns); ax.set_yticks(range(H.shape[0])); ax.set_yticklabels(H.index,fontsize=8)
for i in range(H.shape[0]):
    for j in range(H.shape[1]): ax.text(j,i,f'{H.values[i,j]:.1f}',ha='center',va='center',fontsize=7)
ax.set_title('Feature means by surface (z-scored)',fontweight='bold'); fig.colorbar(im,fraction=0.046,pad=0.04)
plt.tight_layout(); plt.savefig(os.path.join(CHARTS_DIR,'groundup_feature_surface_heatmap.png'),dpi=140,bbox_inches='tight'); plt.show()
# (C) scales
SOFT={'grass','soft'}; COMPL=[c for c in ['accel_jerk_ratio','accel_hf','accel_spec_centroid','accel_zcr','gyro_hf','stance_frac','step_dur'] if c in full.columns]
hard_y=(~full['surface'].isin(SOFT)).astype(int).values; grp=full['source_recording'].values
oof=np.zeros(len(full)); Xc2=full[COMPL].fillna(0).values
if len(np.unique(hard_y))==2:
    for tr,te in StratifiedGroupKFold(N_SPLITS,shuffle=True,random_state=42).split(Xc2,hard_y,grp):
        p=Pipeline([('s',StandardScaler()),('lr',LogisticRegression(max_iter=1000,class_weight='balanced'))]); p.fit(Xc2[tr],hard_y[tr]); oof[te]=p.predict_proba(Xc2[te])[:,1]
    full['hardness']=oof*100
else: full['hardness']=50.0
EVEN_NEG=[c for c in ['cop_wander','press_jitter','tiltA_s2s','tiltB_s2s','gyro_peak_norm'] if c in full.columns]
ze=((full[EVEN_NEG]-full[EVEN_NEG].mean())/full[EVEN_NEG].std()).clip(-4,4); full['evenness']=(-ze).mean(1).rank(pct=True)*100
print('\nMean scale position by surface:'); print(full.groupby('surface')[['hardness','evenness']].mean().round(1).to_string())
full[['surface','source_user','hardness','evenness']].to_csv(os.path.join(CHARTS_DIR,'groundup_scales_scored.csv'),index=False)
# (D) populations
fig=plt.figure(figsize=(15,4.6)); gs=fig.add_gridspec(1,3,width_ratios=[1,1,1.2]); cl=list(classes)
for k,sca in enumerate(['hardness','evenness']):
    ax=fig.add_subplot(gs[0,k]); data=[full[full.surface==sname][sca].values for sname in cl]
    parts=ax.violinplot(data,showmeans=True,showextrema=False)
    for pc,sname in zip(parts['bodies'],cl): pc.set_facecolor(SURFACE_COLORS.get(sname,'#9E9E9E')); pc.set_alpha(.7)
    ax.set_xticks(range(1,len(cl)+1)); ax.set_xticklabels(cl); ax.set_ylabel(f'{sca} (0-100)'); ax.set_title(f'{sca.capitalize()} by surface',fontweight='bold'); ax.grid(axis='y',alpha=.3)
ax=fig.add_subplot(gs[0,2])
for sname in cl:
    d=full[full.surface==sname].sample(min(400,int((full.surface==sname).sum())),random_state=1)
    ax.scatter(d.hardness,d.evenness,s=8,alpha=.35,color=SURFACE_COLORS.get(sname,'#9E9E9E'),label=sname)
    ax.scatter(full[full.surface==sname].hardness.mean(),full[full.surface==sname].evenness.mean(),s=320,color=SURFACE_COLORS.get(sname,'#9E9E9E'),edgecolor='k',marker='X',zorder=5)
ax.set_xlabel('hardness = P(firm)'); ax.set_ylabel('evenness'); ax.set_title('Surfaces in hardness x evenness space',fontweight='bold'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(CHARTS_DIR,'groundup_scales_populations.png'),dpi=140,bbox_inches='tight'); plt.show()